# Historical VaR

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Set up parameters
TICKER = 'GOOG'
CONFIDENCE_LEVEL = 0.99
LOOKBACK_DAYS = 251  # ~1 year of trading days
HORIZON_DAYS = 10
PORTFOLIO_VALUE = 1_000_000

print(f"Analyzing {TICKER} with {CONFIDENCE_LEVEL:.0%} confidence")
print(f"Lookback period: {LOOKBACK_DAYS} days, Horizon: {HORIZON_DAYS} days")
print(f"Portfolio value: ${PORTFOLIO_VALUE:,}")

Analyzing GOOG with 99% confidence
Lookback period: 251 days, Horizon: 10 days
Portfolio value: $1,000,000


## 1. Fetch Prices 

In [3]:
def fetch_prices(ticker, lookback, var_date=None):
    """Fetch daily close prices for a ticker.

    Gets the last `lookback` trading days of data up to the day before `var_date`.
    If `var_date` is not given, it uses the last business day.
    """
    # if the var date is none fetch the last business day
    if var_date is None:
        var_date = (pd.Timestamp.today() - pd.offsets.BDay()).date()
    
    calendar_days = int(lookback * 1.6)
    start = var_date - pd.Timedelta(days=calendar_days)

    df = yf.download(
        ticker,
        start=start.strftime("%Y-%m-%d"),
        end=var_date.strftime("%Y-%m-%d"),
        progress=False,
        interval="1d",
        auto_adjust=True
    )

    if df.empty:
        raise ValueError(f"No data returned for ticker '{ticker}'.")
    
    prices = df["Close"].squeeze()
    prices.name = ticker
    result = prices.tail(lookback)
    return result

In [4]:
prices = fetch_prices(TICKER, LOOKBACK_DAYS)
print(f"Shape: {prices.shape}")
print(f"Start date: {prices.index.min().date()}")
print(f"End date: {prices.index.max().date()}")

Shape: (251,)
Start date: 2025-03-26
End date: 2026-03-25


## 2. Return Calculation

Calculate daily returns from the price data.

In [ ]:
def compute_returns(prices, kind="arithmetic"):
    """Compute daily returns from a price series.

    kind : "arithmetic" or "log"
        arithmetic  ->  (P_t - P_{t-1}) / P_{t-1}
        log         ->  log(P_t) - log(P_{t-1})
    """
    if kind == "log":
        returns = np.log(prices) - np.log(prices.shift(1))
        returns.name = "Daily Log Return"
    else:
        returns = (prices - prices.shift(1)) / prices.shift(1)
        returns.name = "Daily Return"
    return returns.dropna()

In [ ]:
# Calculate returns
daily_returns = compute_returns(prices, kind="arithmetic")

print("Daily return series:")
print("Shape: ", daily_returns.shape)
print("\n ", daily_returns.head())

## 3. Calculate VaR and ES

In [ ]:
def calculate_historical_var(returns, confidence):
    """Compute VaR from returns using the percentile method.

    VaR is the (1 - confidence) percentile of returns, negated to express as loss.
    Returns VaR as a positive loss fraction.
    """
    vals = returns.values
    return -float(np.percentile(np.asarray(vals), (1.0 - confidence) * 100))


def calculate_historical_es(returns, confidence):
    """Compute ES from returns using the percentile method.

    ES = E[loss | loss > VaR], the mean of losses exceeding VaR.
    Returns ES as a positive loss fraction.
    """
    var = calculate_historical_var(returns, confidence)
    losses = -np.asarray(returns.values)
    tail = losses[losses > var]
    return float(np.mean(tail)) if len(tail) > 0 else var

In [ ]:
var_pct = calculate_historical_var(daily_returns, CONFIDENCE_LEVEL)
es_pct = calculate_historical_es(daily_returns, CONFIDENCE_LEVEL)
print(f"VaR: {var_pct:.4f} ({var_pct*100:.2f}%)")
print(f"ES: {es_pct:.4f} ({es_pct*100:.2f}%)")

## 5. Orchestration of the workflow

In [ ]:
def historical_var_es_pipeline(ticker, confidence, lookback, n_days, portfolio_value, end_date=None):
    """Run the full historical VaR workflow.

    Fetches data, computes 1-day VaR/ES, and scales results to n-day horizon.
    Returns dollar VaR/ES along with the underlying data.
    """
    # 1. Fetch data and compute returns
    prices = fetch_prices(ticker, lookback, end_date)
    daily_returns = compute_returns(prices, kind="arithmetic")

    # 2. Calculate 1-day VaR and ES
    var_1d_pct = calculate_historical_var(daily_returns, confidence)
    es_1d_pct = calculate_historical_es(daily_returns, confidence)
    var_1d = var_1d_pct * portfolio_value
    es_1d = es_1d_pct * portfolio_value

    # 3. Scale to N-day horizon
    scaling_factor = np.sqrt(n_days)
    var_nd = var_1d * scaling_factor
    es_nd = es_1d * scaling_factor

    return {
        "var_1d": var_1d,
        "var_nd": var_nd,
        "es_1d": es_1d,
        "es_nd": es_nd,
        "prices": prices,
        "daily_returns": daily_returns,
    }

In [ ]:
results = historical_var_es_pipeline(
    ticker=TICKER,
    confidence=CONFIDENCE_LEVEL,
    lookback=LOOKBACK_DAYS,
    n_days=HORIZON_DAYS,
    portfolio_value=PORTFOLIO_VALUE)

In [ ]:
print("=" * 60)
print(f"HISTORICAL VaR ANALYSIS SUMMARY - {TICKER}")
print("=" * 60)
print(f"VaR Date: {datetime.now().strftime('%Y-%m-%d')}")
print(f"Portfolio Value: ${PORTFOLIO_VALUE:,}")
print(f"Confidence Level: {CONFIDENCE_LEVEL:.0%}")
print(f"Time Horizon: {HORIZON_DAYS} days")
print(f"Historical Period: {LOOKBACK_DAYS} trading days")
print()
print("VaR METRICS:")
print('-'*60)
print(f"  {HORIZON_DAYS}-Day VaR: ${results['var_nd']:,.2f} ({results['var_nd']/PORTFOLIO_VALUE*100:.2f}%)")
print(f"  {HORIZON_DAYS}-Day ES:  ${results['es_nd']:,.2f} ({results['es_nd']/PORTFOLIO_VALUE*100:.2f}%)")
print()